# TP2 — Prévision de la consommation électrique (BiLSTM + Attention)

**Objectif :** prédire la consommation électrique horaire (`Global_active_power`) à partir de
l'historique des 48 dernières heures, sur le dataset **UCI Household Power Consumption**.

**Approche :** réseau **BiLSTM (bidirectionnel, 2 couches)** couplé à un **mécanisme d'attention**
dot-product, entraîné avec une **Huber loss** (robuste aux pics de consommation).

> Notebook conçu pour s'exécuter directement sur **Google Colab**.

## Cellule 1 — Installation des dépendances

Installation des bibliothèques nécessaires (à exécuter sur Colab).

In [ ]:
!pip install torch torchvision numpy pandas matplotlib seaborn scikit-learn requests

## Cellule 2 — Téléchargement automatique du dataset UCI

On télécharge et décompresse l'archive UCI Household Power Consumption directement dans le répertoire courant.

In [ ]:
# Téléchargement et extraction de l'archive ZIP du dataset UCI
import requests, zipfile, io

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00235/household_power_consumption.zip"
r = requests.get(url)
z = zipfile.ZipFile(io.BytesIO(r.content))
z.extractall(".")  # extrait household_power_consumption.txt dans le dossier courant
print("Dataset extrait :", z.namelist())

## Cellule 3 — Imports et configuration

Imports des bibliothèques et fixation de la graine aléatoire (`seed=42`).

In [ ]:
# Bibliothèques de base
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Outils sklearn
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Fixation de la graine pour la reproductibilité
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {device}")

## Cellule 4 — Chargement et nettoyage

- Parsing de `Date` + `Time` en index `datetime`.
- Remplacement des `?` par `NaN` puis **interpolation linéaire**.
- Cible : `Global_active_power`.
- **Rééchantillonnage horaire** (moyenne).

In [ ]:
# Chargement du fichier : séparateur ';', les '?' marquent les valeurs manquantes
df = pd.read_csv(
    "household_power_consumption.txt",
    sep=";",
    na_values=["?"],          # '?' -> NaN
    low_memory=False,
)

# Construction d'un index datetime à partir des colonnes Date et Time
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], format="%d/%m/%Y %H:%M:%S")
df = df.set_index("datetime").drop(columns=["Date", "Time"])

# Conversion en numérique (les colonnes sont lues comme texte à cause des '?')
df = df.apply(pd.to_numeric, errors="coerce")

# Interpolation linéaire des valeurs manquantes
df = df.interpolate(method="linear").bfill().ffill()

print(f"Dimensions : {df.shape}")
df.head()

In [ ]:
# Rééchantillonnage horaire : on garde la moyenne de Global_active_power par heure
hourly = df[["Global_active_power"]].resample("1H").mean()
hourly = hourly.interpolate(method="linear")  # comble d'éventuels trous après resample

print(f"Série horaire : {hourly.shape}")
hourly.head()

## Cellule 5 — Feature engineering

On ajoute des variables temporelles cycliques (encodage sin/cos pour l'heure, le jour de la
semaine et le mois) ainsi qu'une **moyenne glissante sur 24h**.

In [ ]:
# Encodage cyclique : sin/cos pour capturer la périodicité sans rupture (23h -> 0h)
idx = hourly.index
hourly["hour_sin"] = np.sin(2 * np.pi * idx.hour / 24)
hourly["hour_cos"] = np.cos(2 * np.pi * idx.hour / 24)
hourly["dow_sin"] = np.sin(2 * np.pi * idx.dayofweek / 7)
hourly["dow_cos"] = np.cos(2 * np.pi * idx.dayofweek / 7)
hourly["month_sin"] = np.sin(2 * np.pi * (idx.month - 1) / 12)
hourly["month_cos"] = np.cos(2 * np.pi * (idx.month - 1) / 12)

# Moyenne glissante 24h de la consommation (tendance journalière)
hourly["roll_mean_24h"] = hourly["Global_active_power"].rolling(window=24, min_periods=1).mean()

hourly.head()

## Cellule 6 — Normalisation et fenêtrage

- **MinMaxScaler** sur toutes les features (fit sur le train uniquement).
- **Fenêtre glissante** de 48h pour prédire la consommation de l'heure suivante (h+1).
- Split **70% / 15% / 15%** (train / val / test) en respectant l'ordre temporel (pas de shuffle).

In [ ]:
# Ordre des colonnes : la cible (Global_active_power) est en première position (index 0)
feature_cols = ["Global_active_power", "hour_sin", "hour_cos", "dow_sin", "dow_cos",
                "month_sin", "month_cos", "roll_mean_24h"]
data = hourly[feature_cols].values.astype(np.float32)
TARGET_IDX = 0  # indice de la colonne cible

# Découpage temporel des indices (sans shuffle)
n = len(data)
train_end = int(0.70 * n)
val_end = int(0.85 * n)

# MinMaxScaler ajusté UNIQUEMENT sur la partie train (évite la fuite d'information)
scaler = MinMaxScaler()
scaler.fit(data[:train_end])
data_scaled = scaler.transform(data)

In [ ]:
WINDOW = 48  # 48h d'historique en entrée

def make_sequences(series, window):
    """Construit les fenêtres glissantes (X) et la cible h+1 (y) à partir d'une série multivariée."""
    X, y = [], []
    for i in range(len(series) - window):
        X.append(series[i:i + window])              # 48h de toutes les features
        y.append(series[i + window, TARGET_IDX])    # consommation à h+1
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)

# Génération des séquences sur l'ensemble de la série
X_all, y_all = make_sequences(data_scaled, WINDOW)

# Découpage train/val/test en respectant l'ordre temporel.
# On décale les bornes de WINDOW car chaque séquence consomme 48h d'historique.
tr_end = train_end - WINDOW
va_end = val_end - WINDOW
X_train, y_train = X_all[:tr_end], y_all[:tr_end]
X_val, y_val = X_all[tr_end:va_end], y_all[tr_end:va_end]
X_test, y_test = X_all[va_end:], y_all[va_end:]

print(f"Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}")

# DataLoaders PyTorch
BATCH_SIZE = 64
train_dl = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=BATCH_SIZE, shuffle=True)
val_dl = DataLoader(TensorDataset(torch.tensor(X_val), torch.tensor(y_val)), batch_size=BATCH_SIZE, shuffle=False)
test_dl = DataLoader(TensorDataset(torch.tensor(X_test), torch.tensor(y_test)), batch_size=BATCH_SIZE, shuffle=False)
n_features = X_train.shape[2]